# Adquisición de Datos: APIs, scraping y el desorden del mundo real

**Ciencia de Datos, Sección A** · Sesión 4 · 30 de julio de 2026

Notebook companion de la presentación. Corran cada celda con `Shift+Enter`.

Dependencias: `pip install requests pandas beautifulsoup4 lxml`

Las celdas de API y scraping requieren conexión a internet.

## 1. HTTP en cinco minutos

Su programa manda un **request**, un servidor devuelve un **response**. Códigos que importan: `200` (bien), `404` (no existe), `429` (van muy rápido), `500` (falló el servidor).

## 2. APIs REST con requests

Open-Meteo es una API pública y gratuita de clima, sin llave. Usamos las coordenadas de Ciudad de Guatemala.

In [1]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 14.63,
    "longitude": -90.51,
    "daily": "temperature_2m_max",
    "timezone": "America/Guatemala",
}
r = requests.get(url, params=params, timeout=10)
print(r.status_code)
datos = r.json()
print(datos.keys())

200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


In [2]:
import pandas as pd

clima = pd.DataFrame(datos["daily"])
clima["time"] = pd.to_datetime(clima["time"])
print(clima.head())

# cuando el JSON viene anidado de verdad:
plano = pd.json_normalize(datos, sep="_")
print(plano.columns.tolist())

        time  temperature_2m_max
0 2026-07-30                27.1
1 2026-07-31                27.6
2 2026-08-01                28.4
3 2026-08-02                28.9
4 2026-08-03                28.1
['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units_time', 'daily_units_temperature_2m_max', 'daily_time', 'daily_temperature_2m_max']


In [3]:
base = "https://api.worldbank.org/v2"
ruta = "/country/GTM/indicator/NY.GDP.PCAP.CD"

r = requests.get(base + ruta,
                 params={"format": "json", "per_page": 100},
                 timeout=10)
meta, filas = r.json()   # la respuesta trae 2 elementos
pib = pd.json_normalize(filas)
pib[["date", "value"]].head()

,date,value
0,2025,6598.180330
1,2024,6150.894653
2,2023,5754.428339
3,2022,5356.924359
4,2021,4912.622404


In [4]:
import os, time

# La llave NUNCA va escrita en el codigo
# llave = os.environ["MI_API_KEY"]
# headers = {"Authorization": f"Bearer {llave}"}

# Paginacion: pedir de a poco, acumular
# todo = []
# for pagina in range(1, 6):
#     r = requests.get(url, params={"page": pagina},
#                      headers=headers, timeout=10)
#     if r.status_code != 200:
#         break
#     todo.extend(r.json())
#     time.sleep(0.5)   # cortesia con el servidor

**Higiene al consumir una API**

- Siempre `timeout`: sin él, su script puede colgarse para siempre
- Siempre revisar `status_code` antes de leer `.json()`
- Guardar la respuesta cruda en disco antes de transformarla
- Nunca subir llaves a GitHub: variables de entorno o `.env` ignorado

In [5]:
import json

with open("clima_crudo.json", "w", encoding="utf-8") as f:
    json.dump(datos, f, ensure_ascii=False, indent=2)

print("respuesta cruda guardada")

respuesta cruda guardada


## 3. Web scraping

Una página es un árbol de etiquetas con atributos. Scrapear es navegar ese árbol y sacar el texto de los nodos correctos.

In [6]:
from bs4 import BeautifulSoup

html_ejemplo = """
<table class="wikitable">
  <tr><th>Departamento</th><th>Poblacion</th></tr>
  <tr><td>Guatemala</td><td>3015081</td></tr>
  <tr><td>Huehuetenango</td><td>1170669</td></tr>
</table>
"""

sopa = BeautifulSoup(html_ejemplo, "html.parser")
print(sopa.find("table", class_="wikitable") is not None)
print([td.get_text(strip=True) for td in sopa.select("table.wikitable td")])

True
['Guatemala', '3015081', 'Huehuetenango', '1170669']


In [7]:
import io

url = ("https://es.wikipedia.org/wiki/"
       "Departamentos_de_Guatemala")

# read_html con la URL directa: Wikipedia responde 403 al agente
# por defecto, y pasarle el HTML como cadena esta deprecado.
# El patron moderno: bajar con requests y envolver en StringIO.
cab = {"User-Agent": "curso-ds-ufm/1.0"}
resp = requests.get(url, headers=cab, timeout=20)
tablas = pd.read_html(io.StringIO(resp.text))   # lista de DataFrames
print(len(tablas))
tablas[0].head()

48


,Departamentos de Guatemala,Departamentos de Guatemala.1,Departamentos de Guatemala.2
0,NaN,NaN,NaN
1,País,Guatemala,Guatemala
2,Categoría,Primer nivel de división administrativa,Primer nivel de división administrativa
3,Datos estadísticos,Datos estadísticos,Datos estadísticos
4,Número actual,22,22


## 4. Datos sucios del mundo real

Encodings, fechas mixtas, duplicados y centinelas: los cuatro sospechosos.

In [8]:
# Si ven "Ã±" donde deberia haber "ñ", leyeron latin-1 como utf-8.
# df = pd.read_csv("municipios.csv", encoding="latin-1")
# df = pd.read_csv("municipios.csv", encoding="utf-8",
#                  encoding_errors="replace")

sucio = pd.DataFrame({
    "id": [1, 2, 2, 3],
    "ciudad": [" Guatemala ", "xela", "xela", "GUATE"],
    "fecha": ["2026-01-05", "05/02/2026", "05/02/2026", "no aplica"],
    "valor": [10.0, -999, -999, 25.0],
})
sucio

,id,ciudad,fecha,valor
0,1,Guatemala,2026-01-05,10.0
1,2,xela,05/02/2026,-999.0
2,2,xela,05/02/2026,-999.0
3,3,GUATE,no aplica,25.0


In [9]:
limpio = sucio.copy()
limpio["fecha"] = pd.to_datetime(limpio["fecha"], errors="coerce",
                                 format="mixed", dayfirst=True)
print(limpio["fecha"].isna().sum())

print(limpio.duplicated().sum())
limpio = limpio.drop_duplicates(subset=["id"])

limpio = limpio.replace([-999, "N/A", "", "NULL"], pd.NA)
limpio["ciudad"] = limpio["ciudad"].str.strip().str.lower()
limpio

1
1


,id,ciudad,fecha,valor
0,1,guatemala,2026-05-01,10.0
1,2,xela,2026-02-05,<NA>
3,3,guate,NaT,25.0


## 5. Ética y procedencia

Que se pueda no significa que se deba: `robots.txt`, términos de uso, datos personales. Y siempre documentar de dónde salió el dato y cuándo.

In [10]:
procedencia = {
    "fuente": "Open-Meteo API v1",
    "url": url,
    "parametros": params,
    "fecha_descarga": "2026-07-30",
    "filas": len(clima),
    "licencia": "CC BY 4.0",
}

with open("procedencia.json", "w", encoding="utf-8") as f:
    json.dump(procedencia, f, ensure_ascii=False, indent=2)

procedencia

{'fuente': 'Open-Meteo API v1',
 'url': 'https://es.wikipedia.org/wiki/Departamentos_de_Guatemala',
 'parametros': {'latitude': 14.63,
  'longitude': -90.51,
  'daily': 'temperature_2m_max',
  'timezone': 'America/Guatemala'},
 'fecha_descarga': '2026-07-30',
 'filas': 7,
 'licencia': 'CC BY 4.0'}

## 6. Ejercicios

Completen los `# ¿Que va aqui?`.

### Ejercicio 1: una API a DataFrame

In [20]:
url = "https://api.open-meteo.com/v1/forecast"
params = {"latitude": 14.63, "longitude": -90.51,
          "daily": "temperature_2m_max,precipitation_sum",
          "timezone": "America/Guatemala"}

# a) hagan el GET y verifiquen status_code

r = requests.get(url, params=params, timeout=10)
print(r.status_code)
datos = r.json()
print(datos.keys())

# b) construyan un DataFrame de datos["daily"]

import numpy as np
import pandas as pd

df = pd.DataFrame({
    "time": datos["daily"]["time"],
    "temperature_2m_max": datos["daily"]["temperature_2m_max"]
})
print(df)
print(df.dtypes)   # cada columna con su propio dtype

# c) conviertan "time" a datetime

df["time"] = pd.to_datetime(df["time"])

# d) ¿que dia se espera mas lluvia?

df = pd.DataFrame({
    "precipitation_sum": datos["daily"]["precipitation_sum"]
})
print(df)
print(df.dtypes)

# ¿Que va aqui?

200
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
         time  temperature_2m_max
0  2026-07-30                27.1
1  2026-07-31                27.6
2  2026-08-01                28.4
3  2026-08-02                28.9
4  2026-08-03                28.1
5  2026-08-04                26.8
6  2026-08-05                24.4
time                      str
temperature_2m_max    float64
dtype: object
   precipitation_sum
0                1.8
1                1.2
2                0.0
3                0.0
4                1.1
5                4.9
6                1.0
precipitation_sum    float64
dtype: object


### Ejercicio 2: scrapear, limpiar y documentar

In [12]:
url = ("https://es.wikipedia.org/wiki/"
       "Departamentos_de_Guatemala")

# a) bajen la pagina con requests (User-Agent propio) y usen
#    pd.read_html(io.StringIO(resp.text)); elijan la correcta
# b) renombren columnas a nombres cortos y limpios
# c) quiten comas y espacios; a numerico con
#    pd.to_numeric(..., errors="coerce")
# d) ordenen por poblacion y saquen el top 5
# e) escriban la ficha de procedencia (fuente, url, fecha,
#    filas, licencia) y guardenla como JSON junto al CSV limpio

# ¿Que va aqui?

# Verificacion: dtypes debe mostrar int o float
# print(df.dtypes)